# Stage 6 & 7: Algorithm 4 — Multi-Layer Perceptron (MLP Neural Network)

**Owner:** Member D  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/mlp_tuned.pkl`  

## Scope
The Multi-Layer Perceptron (MLP) is a **feedforward neural network** that learns non-linear
combinations of input features through multiple hidden layers with ReLU activation.

Key advantages:
- Captures complex non-linear feature interactions that linear models cannot
- L2 regularization (`alpha`) prevents overfitting on the 67-feature input space
- Early stopping prevents training beyond convergence
- Works well with StandardScaler-normalized inputs (already applied in data/final/)

This notebook:
1. Loads the final modeling datasets from `data/final/`
2. Trains a default MLP baseline
3. Performs Stratified K-Fold hyperparameter tuning on `X_train` only
4. Evaluates the tuned model on `X_val` with full metrics
5. Exports the tuned model artifact


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)

FINAL_DIR  = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train : {X_train.shape} | Late rate: {y_train.mean()*100:.2f}%')
print(f'X_val   : {X_val.shape}   | Late rate: {y_val.mean()*100:.2f}%')
print('Note: X_train is already StandardScaler-normalized (fitted on train only in Notebook 05).')


X_train : (67529, 67) | Late rate: 9.03%
X_val   : (14470, 67)   | Late rate: 5.34%
Note: X_train is already StandardScaler-normalized (fitted on train only in Notebook 05).


## 2. Default MLP Baseline
An untuned MLP with default architecture establishes the baseline before hyperparameter search.

In [2]:
mlp_base = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, early_stopping=True, random_state=42)
mlp_base.fit(X_train, y_train)
base_probs = mlp_base.predict_proba(X_val)[:, 1]

print('=== DEFAULT MLP BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, base_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, base_probs):.4f}')


=== DEFAULT MLP BASELINE ===
Val ROC-AUC : 0.7565
Val PR-AUC  : 0.1557


## 3. Hyperparameter Tuning — Stratified GridSearchCV
**Parameters tuned:**
- `hidden_layer_sizes`: Network architecture. `(64, 32)` = 2 hidden layers with 64 and 32 neurons.
- `alpha`: L2 regularization penalty. Prevents weights from growing too large (overfitting).
- `activation`: `relu` (Rectified Linear Unit) is the standard for hidden layers.

**Note:** `early_stopping=True` stops training if validation loss stops improving,
preventing unnecessary computation and overfitting.

In [3]:
param_grid = {
    'hidden_layer_sizes': [(64, 32), (128, 64)],
    'alpha': [0.001, 0.01],
    'activation': ['relu']
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    MLPClassifier(max_iter=100, early_stopping=True, random_state=42),
    param_grid, scoring='average_precision', cv=cv, n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train)

print(f'Best Parameters : {grid.best_params_}')
print(f'Best CV PR-AUC  : {grid.best_score_:.4f}')
best_mlp = grid.best_estimator_


Fitting 3 folds for each of 4 candidates, totalling 12 fits


Best Parameters : {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (64, 32)}
Best CV PR-AUC  : 0.2491


## 4. Validation Set Evaluation

In [4]:
val_probs = best_mlp.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print('=== TUNED MLP NEURAL NET — VALIDATION SET METRICS ===')
print(f'Val ROC-AUC        : {roc_auc_score(y_val, val_probs):.4f}')
print(f'Val PR-AUC         : {average_precision_score(y_val, val_probs):.4f}')
print(f'Val F1-Score (0.5) : {f1_score(y_val, val_preds):.4f}')
print(f'Val Recall   (0.5) : {recall_score(y_val, val_preds):.4f}')
print(f'Val Precision(0.5) : {precision_score(y_val, val_preds):.4f}')

print('\nClassification Report:')
print(classification_report(y_val, val_preds, target_names=['On-Time (0)', 'Late (1)']))

cm = confusion_matrix(y_val, val_preds)
print(f'Confusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')


=== TUNED MLP NEURAL NET — VALIDATION SET METRICS ===
Val ROC-AUC        : 0.7766
Val PR-AUC         : 0.1773
Val F1-Score (0.5) : 0.0127
Val Recall   (0.5) : 0.0065
Val Precision(0.5) : 0.3846

Classification Report:
              precision    recall  f1-score   support

 On-Time (0)       0.95      1.00      0.97     13697
    Late (1)       0.38      0.01      0.01       773

    accuracy                           0.95     14470
   macro avg       0.67      0.50      0.49     14470
weighted avg       0.92      0.95      0.92     14470

Confusion Matrix:
  TN=13,689  FP=8
  FN=768  TP=5


## 5. Training Convergence Information
Verify how many iterations the MLP took to converge with early stopping.

In [5]:
print(f'Network Architecture  : {best_mlp.hidden_layer_sizes}')
print(f'Alpha (L2 penalty)    : {best_mlp.alpha}')
print(f'Activation function   : {best_mlp.activation}')
print(f'Iterations trained    : {best_mlp.n_iter_}')
print(f'Number of layers      : {best_mlp.n_layers_}')
print(f'Loss at convergence   : {best_mlp.loss_:.6f}')


Network Architecture  : (64, 32)
Alpha (L2 penalty)    : 0.001
Activation function   : relu
Iterations trained    : 16
Number of layers      : 4
Loss at convergence   : 0.253231


## 6. Save Tuned Model Artifact

In [6]:
joblib.dump(best_mlp, MODELS_DIR / 'mlp_tuned.pkl')
print('Saved: models/mlp_tuned.pkl')
print(f'Architecture : {best_mlp.hidden_layer_sizes}')
print(f'Alpha        : {best_mlp.alpha}')


Saved: models/mlp_tuned.pkl
Architecture : (64, 32)
Alpha        : 0.001
